# Análisis de un grupo de Telegram

En esta libreta analizo el historial de un grupo público de Telegram en ruso, dedicado a
la compraventa e intercambio de cuentas de un videojuego, entre agosto de 2025 y
septiembre de 2026. Los datos crudos no se suben al repositorio, así que la libreta se
entrega ya ejecutada. Para proteger a los participantes, los nombres se reemplazaron por
superhéroes y solo se muestran resultados agregados, nunca mensajes completos.

## 0. Configuración

Cargo las librerías y las rutas del proyecto, que están definidas en `src/config.py`.

In [ ]:
import re
import sys
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from lxml import html

# Para poder importar src/ desde la carpeta notebooks/
PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJ_ROOT))

from src.config import CHAT_EXPORT_DIR, FIGURES_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR

pd.set_option("display.max_columns", 30)

# Estilo de las gráficas
sns.set_theme(style="whitegrid")
AZUL = "#2a78d6"  # color principal
GRIS = "#898781"  # "Otros" y líneas de referencia

## 1. Lectura del HTML exportado

Telegram exportó el chat en 35 archivos HTML (`messages.html`, `messages2.html`, ...,
`messages35.html`). Los leo en orden numérico para conservar la secuencia de los mensajes.

In [ ]:
def numero_archivo(ruta):
    # messages.html es el 1, messages2.html el 2, etc.
    return int(re.search(r"messages(\d*)\.html", ruta.name).group(1) or 1)


archivos = sorted(CHAT_EXPORT_DIR.glob("messages*.html"), key=numero_archivo)
print(f"{len(archivos)} archivos: {archivos[0].name} ... {archivos[-1].name}")

Cada mensaje de usuario es un bloque `div` con las clases `message default`; los avisos
del sistema, como los separadores de fecha, se ignoran. La multimedia no se descargó, así
que de cada foto, video o sticker solo conozco su tipo y, en los stickers, el emoji
asociado. En audios y archivos adjuntos el título es el nombre de la canción o del
archivo, por eso el tipo se deduce de la clase del bloque y ese título no se guarda.
Cuando alguien menciona a otra persona que no tiene `@usuario`, Telegram escribe su nombre
real en el texto; esas menciones se reemplazan desde aquí por `@usuario_anon`.

In [ ]:
# Tipos que Telegram escribe como título del bloque multimedia
TITULOS_MEDIA = {"Photo", "Sticker", "Video file", "Animation", "Voice message", "Video message"}


def hijo(elemento, clases):
    # Primer hijo directo que tiene todas las clases indicadas
    buscadas = set(clases.split())
    for h in elemento:
        if buscadas <= set((h.get("class") or "").split()):
            return h
    return None


def extraer_texto(div_texto):
    # Los saltos de línea vienen como <br>
    for br in div_texto.iter("br"):
        br.tail = "\n" + (br.tail or "")
    # Menciones sin @usuario: el enlace muestra el nombre real de la persona
    for a in div_texto.xpath('.//a[contains(@onclick, "ShowMentionName")]'):
        a.text = "@usuario_anon"
    return div_texto.text_content().strip()


def tipo_de_media(media):
    if media.find_class("media_poll"):
        return "Poll"
    titulo = media.find_class("title")[0].text_content().strip()
    if titulo in TITULOS_MEDIA:
        return titulo
    if media.find_class("media_audio_file"):
        return "Audio file"
    if media.find_class("media_file"):
        return "File"
    return "Other"


def contar_reacciones(body):
    total = 0
    for reaccion in body.find_class("reaction"):
        conteo = reaccion.find_class("count")
        if conteo:
            total += int(conteo[0].text_content())
        else:
            # Con pocas reacciones se muestran avatares en vez de un número
            total += len(reaccion.find_class("userpic"))
    return total


def leer_mensaje(msg):
    body = hijo(msg, "body")
    # En los reenviados, el texto y la multimedia están dentro de otro bloque
    reenviado = hijo(body, "forwarded body")
    contenido = reenviado if reenviado is not None else body

    nombre = hijo(body, "from_name")
    texto = hijo(contenido, "text")
    media = hijo(contenido, "media_wrap")
    respuesta = hijo(body, "reply_to")

    tipo_media = tipo_de_media(media) if media is not None else None
    sticker_emoji = None
    if tipo_media == "Sticker":
        # El estado del sticker se ve así: "<emoji>, <tamaño>"
        sticker_emoji = media.find_class("status")[0].text_content().split(",")[0].strip()
    responde_a_id = None
    if respuesta is not None:
        # El enlace es "#go_to_message33351" o "messages2.html#go_to_message33351"
        href = respuesta.find(".//a").get("href")
        responde_a_id = int(href.split("go_to_message")[1])

    return {
        "id_mensaje": int(msg.get("id").replace("message", "")),
        "fecha": hijo(body, "date").get("title"),
        "nombre": nombre.text_content().strip() if nombre is not None else None,
        "es_continuacion": "joined" in msg.get("class").split(),
        "reenviado": reenviado is not None,
        "es_respuesta": respuesta is not None,
        "responde_a_id": responde_a_id,
        "texto": extraer_texto(texto) if texto is not None else None,
        "tipo_media": tipo_media,
        "sticker_emoji": sticker_emoji,
        "n_reacciones": contar_reacciones(body),
    }

Los mensajes seguidos de una misma persona no repiten el nombre, así que lo relleno con
el último remitente visto. Algunos nombres aparecen como "Nombre via @bot"; me quedo con
el nombre y marco esos mensajes en la columna `via_bot`. Las horas vienen en UTC-07:00, la
zona horaria de la computadora con la que se exportó el chat; en la tabla tidy se pasan a
hora de Moscú.

In [ ]:
filas = []
for ruta in archivos:
    raiz = html.parse(str(ruta)).getroot()
    for msg in raiz.find_class("message"):
        if "default" in msg.get("class").split():
            fila = leer_mensaje(msg)
            fila["archivo"] = numero_archivo(ruta)
            filas.append(fila)

df_crudo = pd.DataFrame(filas)
df_crudo["timestamp"] = pd.to_datetime(df_crudo["fecha"], format="%d.%m.%Y %H:%M:%S UTC%z")
df_crudo["remitente"] = df_crudo["nombre"].ffill()
df_crudo["via_bot"] = df_crudo["remitente"].str.contains(" via @", regex=False)
df_crudo["remitente"] = df_crudo["remitente"].str.replace(r" via @\w+$", "", regex=True)
df_crudo["responde_a_id"] = df_crudo["responde_a_id"].astype("Int64")

df_crudo = df_crudo[[
    "archivo", "id_mensaje", "timestamp", "remitente", "via_bot", "es_continuacion",
    "reenviado", "es_respuesta", "responde_a_id", "texto", "tipo_media",
    "sticker_emoji", "n_reacciones",
]]
print(f"{len(df_crudo):,} mensajes leídos")

### Verificación de la lectura

La columna `remitente` todavía tiene los nombres reales, así que no la muestro; se
anonimiza en la sección 2. Por ahora reviso solo conteos generales.

In [ ]:
resumen = pd.Series({
    "Mensajes": len(df_crudo),
    "Remitentes distintos": df_crudo["remitente"].nunique(),
    "Mensajes de 'Deleted Account'": (df_crudo["remitente"] == "Deleted Account").sum(),
    "Con texto": df_crudo["texto"].notna().sum(),
    "Sin texto": df_crudo["texto"].isna().sum(),
    "Con multimedia": df_crudo["tipo_media"].notna().sum(),
    "Reenviados": df_crudo["reenviado"].sum(),
    "Respuestas": df_crudo["es_respuesta"].sum(),
    "Enviados con bot": df_crudo["via_bot"].sum(),
    "Con reacciones": (df_crudo["n_reacciones"] > 0).sum(),
})
print("Del", df_crudo["timestamp"].min(), "al", df_crudo["timestamp"].max())
resumen.to_frame("conteo")

In [ ]:
df_crudo["tipo_media"].value_counts(dropna=False).to_frame("mensajes")

In [ ]:
# Vista previa sin nombres ni textos
df_crudo.drop(columns=["remitente", "texto"]).head()

Se leyeron 34,868 mensajes de 1,322 remitentes. "Deleted Account" junta a todas las
cuentas eliminadas (3,073 mensajes), así que después la excluiré de los rankings por
usuario. Casi un 38 % de los mensajes no tiene texto y la gran mayoría de la multimedia
son fotos, lo que cuadra con un grupo donde se muestran capturas de cuentas en venta.

### Guardado intermedio

Guardo la tabla en `data/interim/` para no tener que volver a leer el HTML. Esa carpeta no se sube a GitHub porque todavía tiene los nombres reales.

In [ ]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_crudo.to_pickle(INTERIM_DATA_DIR / "mensajes_crudos.pkl")

## 2. Anonimización y formato tidy

Antes de cualquier análisis quito todo lo que pueda identificar a una persona: nombres,
menciones, correos, enlaces, tarjetas y teléfonos. Después armo la tabla tidy con la que se responden todas las preguntas.

### Nombres de superhéroes

Cada remitente recibe un nombre de superhéroe según su actividad: el más activo es Batman,
el segundo Superman, y así sucesivamente. Como hay más de 1,300 usuarios y solo 50 nombres,
a partir del usuario 51 se repiten con un sufijo (`Batman_2`, `Superman_2`, ...). "Deleted
Account" no es una persona sino todas las cuentas eliminadas juntas, así que la llamo
"Cuenta eliminada" y no le asigno superhéroe.

In [ ]:
HEROES = [
    "Batman", "Superman", "Flash", "Hulk", "Storm", "Thor", "Robin", "Spiderman",
    "Wolverine", "Aquaman", "Cyclops", "Rogue", "Falcon", "Groot", "Vision",
    "Hawkeye", "Blade", "Raven", "Cyborg", "Gambit", "Nightwing", "Batgirl",
    "Supergirl", "Daredevil", "Elektra", "Punisher", "Shazam", "Atom", "Starfire",
    "Beast", "Iceman", "Colossus", "Nova", "Quicksilver", "Wasp", "Hercules",
    "Magik", "Havok", "Bishop", "Cable", "Domino", "Deadpool", "Sentry",
    "Hellboy", "Zatanna", "Huntress", "Stargirl", "Mera", "Firestorm", "Rocket",
]

# Remitentes ordenados por número de mensajes (en empate, el que escribió primero)
actividad = (
    df_crudo[df_crudo["remitente"] != "Deleted Account"]
    .groupby("remitente")
    .agg(mensajes=("id_mensaje", "size"), primer_mensaje=("id_mensaje", "min"))
    .sort_values(["mensajes", "primer_mensaje"], ascending=[False, True])
)

# La tabla de equivalencias solo existe en memoria; no se guarda en ningún archivo
equivalencias = {}
for i, remitente in enumerate(actividad.index):
    vuelta = i // len(HEROES) + 1
    heroe = HEROES[i % len(HEROES)]
    equivalencias[remitente] = heroe if vuelta == 1 else f"{heroe}_{vuelta}"
equivalencias["Deleted Account"] = "Cuenta eliminada"

print(f"{len(actividad):,} usuarios anonimizados; el sufijo más alto es _{vuelta}")
print("Los 15 más activos:", ", ".join(list(equivalencias.values())[:15]))

### Limpieza del texto

En el texto cambio cada mención `@usuario` por `@usuario_anon` y borro correos, enlaces
(incluidos los de `t.me`, que suelen llevar al perfil de alguien), números de tarjeta y
teléfonos. Como tarjeta cuento 4 grupos de 4 dígitos separados por espacios o guiones, y
como teléfono el formato ruso (`+7` u `8` seguido de 10 dígitos), los números que empiezan
con `+` y cualquier número de 10 dígitos o más. Los números cortos, como precios y
cantidades, se conservan.

In [ ]:
PATRON_CORREO = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
PATRON_ENLACE = (
    r"(?:https?://|www\.)\S+"
    r"|\b(?:t|telegram)\.me/\S+"
    r"|\b[\w-]+\.(?:ru|com|net|org|me|su|gg|io)\b(?:/\S*)?"
)
PATRON_MENCION = r"@\w+"
PATRON_TARJETA = r"\b\d{4}(?:[ -]\d{4}){3}\b"
PATRON_TELEFONO = (
    r"(?:\+7|\b8)[\s(-]*\d{3}[\s)-]*\d{3}[\s-]*\d{2}[\s-]*\d{2}\b"
    r"|\+\d[\d\s()-]{8,}\d"
    r"|\b\d{10,}\b"
)


def limpiar_texto(texto):
    # El correo va antes que los enlaces y las menciones para no dejar la parte del usuario
    texto = texto.str.replace(PATRON_CORREO, " ", regex=True)
    texto = texto.str.replace(PATRON_ENLACE, " ", regex=True, flags=re.IGNORECASE)
    texto = texto.str.replace(PATRON_MENCION, "@usuario_anon", regex=True)
    texto = texto.str.replace(PATRON_TARJETA, " ", regex=True)
    texto = texto.str.replace(PATRON_TELEFONO, " ", regex=True)
    texto = texto.str.replace(r"[ \t]+", " ", regex=True).str.strip()
    # Si el mensaje solo tenía un enlace, queda vacío y lo trato como sin texto
    return texto.mask(texto == "")


texto_limpio = limpiar_texto(df_crudo["texto"])

# Comprobación: no debe quedar ningún dato de contacto ni mención con nombre
revision = pd.Series({
    "Correos": texto_limpio.str.contains(PATRON_CORREO).sum(),
    "Enlaces": texto_limpio.str.contains(PATRON_ENLACE, flags=re.IGNORECASE).sum(),
    "Tarjetas": texto_limpio.str.contains(PATRON_TARJETA).sum(),
    "Teléfonos": texto_limpio.str.contains(PATRON_TELEFONO).sum(),
    "Menciones sin anonimizar": texto_limpio.str.contains(r"@(?!usuario_anon\b)\w+").sum(),
    "Mensajes con menciones anonimizadas": texto_limpio.str.contains("@usuario_anon", regex=False).sum(),
    "Mensajes que quedaron sin texto": (df_crudo["texto"].notna() & texto_limpio.isna()).sum(),
})
revision.to_frame("mensajes")

### Construcción de la tabla tidy

Además del texto limpio, calculo algunas columnas que se usan en las preguntas. La fecha y
hora se pasa de UTC-07:00 a hora de Moscú, que es la referencia más razonable para un grupo
en ruso; en la sección 6 explico por qué esto es un supuesto. El tipo de
contenido se toma de la multimedia; si no hay multimedia pero sí texto, el mensaje es de
texto. Cuento como palabra cualquier secuencia de letras latinas o cirílicas, sin contar números, emojis ni menciones. Por último, marco como
repetido todo texto que aparece idéntico más de 3 veces en el grupo, que casi siempre son
anuncios pegados una y otra vez.

In [ ]:
TIPOS = {
    "Photo": "foto", "Sticker": "sticker", "Animation": "gif",
    "Video file": "video", "Video message": "video", "Voice message": "voz",
}
PATRON_PALABRA = r"[a-zа-яё]+(?:-[a-zа-яё]+)*"


def tipo_contenido(tipo_media, texto):
    if pd.notna(tipo_media):
        return TIPOS.get(tipo_media, "otro")  # audios, archivos y encuestas son "otro"
    if pd.notna(texto):
        return "texto"
    return "otro"  # mensajes que solo tenían un enlace


df = pd.DataFrame({
    "id_mensaje": df_crudo["id_mensaje"],
    "timestamp": df_crudo["timestamp"].dt.tz_convert("Europe/Moscow"),
    "usuario": df_crudo["remitente"].map(equivalencias),
    "texto": texto_limpio,
    "tipo_contenido": [tipo_contenido(m, t) for m, t in zip(df_crudo["tipo_media"], texto_limpio)],
    "reenviado": df_crudo["reenviado"],
    "es_respuesta": df_crudo["es_respuesta"],
    "responde_a_id": df_crudo["responde_a_id"],
    "sticker_emoji": df_crudo["sticker_emoji"],
    "n_reacciones": df_crudo["n_reacciones"],
    "via_bot": df_crudo["via_bot"],
})

veces_texto = df["texto"].map(df["texto"].value_counts())
df["es_repetido"] = veces_texto > 3

sin_menciones = df["texto"].str.replace("@usuario_anon", " ", regex=False)
df["n_palabras"] = sin_menciones.str.count(PATRON_PALABRA, flags=re.IGNORECASE).fillna(0).astype(int)
df["n_emojis"] = df["texto"].fillna("").map(emoji.emoji_count)

print(f"{len(df):,} filas y {df.shape[1]} columnas; usuarios sin nombre: {df['usuario'].isna().sum()}")
# Vista previa sin texto ni usuario: junto con la hora exacta podrían identificar a alguien
df.drop(columns=["texto", "usuario"]).head()

In [ ]:
tipos = df["tipo_contenido"].value_counts().to_frame("mensajes")
tipos["porcentaje"] = (100 * tipos["mensajes"] / len(df)).round(1)
tipos

In [ ]:
pd.Series({
    "Mensajes con texto": df["texto"].notna().sum(),
    "Mensajes repetidos (texto idéntico > 3 veces)": df["es_repetido"].sum(),
    "Textos distintos que se repiten": df.loc[df["es_repetido"], "texto"].nunique(),
    "Palabras en total": df["n_palabras"].sum(),
    "Palabras en mensajes no repetidos": df.loc[~df["es_repetido"], "n_palabras"].sum(),
    "Emojis en total": df["n_emojis"].sum(),
    "Mensajes de 'Cuenta eliminada'": (df["usuario"] == "Cuenta eliminada").sum(),
    "Fotos con descripción": ((df["tipo_contenido"] == "foto") & df["texto"].notna()).sum(),
}).to_frame("conteo")

El 80 % de los mensajes son fotos (muchas con un texto como descripción) y solo el 17.5 %
son mensajes únicamente de texto; stickers, GIFs y audios casi no se usan. De los 21,730
mensajes con texto, 8,404 son repeticiones de apenas 592 textos distintos, y quitarlos deja
160 mil de las 256 mil palabras. Por eso en el análisis de palabras se excluyen los
repetidos, porque si no los anuncios dominarían los resultados.

### ¿Por qué la tabla es tidy?

La tabla cumple las tres reglas de los datos tidy: cada fila es un mensaje, cada columna es
una sola variable y cada celda guarda un único valor. Por ejemplo, el tipo de contenido
está en una sola columna en lugar de tener una columna por tipo, y la fecha y la hora
están juntas en `timestamp`, de donde se pueden sacar el día o la hora cuando se necesiten.
La descripción de cada columna está en el
[diccionario de datos](../references/diccionario_de_datos.md).

El flujo completo es: los HTML de `data/raw/` se leen a una tabla cruda que se guarda en
`data/interim/`; después se anonimiza y limpia, y la tabla tidy resultante se guarda en
`data/processed/`, de donde parte todo el análisis.

### Guardado de la tabla limpia

La tabla tidy se guarda en `data/processed/`. Aunque ya no tiene nombres reales, sí tiene
los textos de los mensajes, así que esa carpeta tampoco se sube a GitHub. A partir de aquí
todo el análisis usa solo `df`.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_pickle(PROCESSED_DATA_DIR / "mensajes_tidy.pkl")
df.dtypes.to_frame("tipo")

## 3. ¿Qué usuario envía más mensajes?

Cuento todos los mensajes de cada usuario, sean fotos, texto u otro contenido. "Cuenta
eliminada" queda fuera del ranking porque junta a muchas personas distintas; solo la muestro
como referencia. Los 15 usuarios más activos aparecen por separado y los demás se agrupan
como "Otros"; esta misma agrupación se usa en las preguntas 2 y 3.

In [ ]:
# Tabla sin "Cuenta eliminada" que se usa en todas las preguntas por usuario
usuarios = df[df["usuario"] != "Cuenta eliminada"].copy()
mensajes_por_usuario = usuarios["usuario"].value_counts()
top15 = mensajes_por_usuario.head(15).index.tolist()
orden = top15 + ["Otros"]
usuarios["grupo"] = usuarios["usuario"].where(usuarios["usuario"].isin(top15), "Otros")


def barras(ax, serie, titulo, etiqueta_x, formato="{:,.0f}"):
    # Barras horizontales ordenadas de arriba hacia abajo, con "Otros" en gris
    colores = [GRIS if nombre == "Otros" else AZUL for nombre in serie.index]
    ax.barh(serie.index, serie.values, color=colores)
    ax.invert_yaxis()
    # Fondo blanco en los números para que se lean aunque una línea pase por detrás
    ax.bar_label(ax.containers[0], fmt=formato, padding=3, fontsize=9,
                 bbox=dict(facecolor="white", edgecolor="none", pad=1))
    ax.set_xlim(0, serie.max() * 1.15)
    ax.set_title(titulo)
    ax.set_xlabel(etiqueta_x)
    ax.grid(axis="y", visible=False)


# Los 15 más activos, "Otros" y "Cuenta eliminada" como referencia
grupo = df["usuario"].where(df["usuario"].isin(top15 + ["Cuenta eliminada"]), "Otros")
tabla_p1 = pd.DataFrame({
    "mensajes": df.groupby(grupo).size(),
    "% del total": 100 * df.groupby(grupo).size() / len(df),
    "% fotos": 100 * (df["tipo_contenido"] == "foto").groupby(grupo).mean(),
    "% repetidos": 100 * df["es_repetido"].groupby(grupo).mean(),
})
tabla_p1.loc[orden + ["Cuenta eliminada"]].round(1)

La gráfica de la izquierda muestra a los 15 más activos. La de la derecha es un boxplot con
todos los usuarios; uso escala logarítmica porque unos pocos usuarios tienen cientos de
mensajes y la mayoría solo unos cuantos, y en escala normal la caja quedaría aplastada
contra el cero.

In [ ]:
print(f"{len(mensajes_por_usuario):,} usuarios; mediana de "
      f"{mensajes_por_usuario.median():.0f} mensajes por usuario")
print(f"Usuarios con un solo mensaje: {(mensajes_por_usuario == 1).sum()}")
print(f"Los 15 más activos enviaron el "
      f"{100 * mensajes_por_usuario.head(15).sum() / len(df):.1f} % de todos los mensajes")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [3, 2]})
barras(ax1, mensajes_por_usuario.head(15), "Los 15 usuarios con más mensajes", "Mensajes")

sns.boxplot(x=mensajes_por_usuario.values, ax=ax2, color=AZUL, log_scale=True, width=0.4)
ax2.set_title(f"Mensajes por usuario ({len(mensajes_por_usuario):,} usuarios)")
ax2.set_xlabel("Mensajes (escala logarítmica)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p1_mensajes_por_usuario.png", dpi=120, bbox_inches="tight")
plt.show()

### Interpretación

Batman es el usuario que más mensajes envía (1,363, el 3.9 % del total), seguido de
Superman (990) y Flash (755), y casi todos sus mensajes son fotos de cuentas. "Cuenta
eliminada" suma todavía más (3,073), pero junta a muchas personas y por eso no cuenta. La
actividad está muy concentrada: la mitad de los 1,321 usuarios envió 5 mensajes o menos y
309 escribieron una sola vez, mientras que los 15 más activos suman casi una cuarta parte
de todo el chat (24.8 %). Además, algunos de los más activos, como Hulk (99.9 %) y Rogue
(83.7 %), publican casi solo anuncios repetidos.

## 4. Palabras promedio por mensaje de texto

Aquí solo cuento los mensajes con al menos una palabra, según la columna `n_palabras` de la
sección 2. Incluyo las descripciones de las fotos, porque en este grupo la mayor parte del
texto acompaña a una captura; los mensajes con solo emojis o números quedan fuera. El
promedio principal incluye los anuncios repetidos, y en la tabla agrego el promedio sin
ellos para comparar.

In [ ]:
con_palabras = usuarios[usuarios["n_palabras"] > 0]
no_repetidos = con_palabras[~con_palabras["es_repetido"]]
solo_texto = con_palabras[con_palabras["tipo_contenido"] == "texto"]

print(f"Promedio del grupo: {con_palabras['n_palabras'].mean():.1f} palabras "
      f"({len(con_palabras):,} mensajes)")
print(f"Sin repetidos: {no_repetidos['n_palabras'].mean():.1f} palabras "
      f"({len(no_repetidos):,} mensajes)")
print(f"Solo mensajes únicamente de texto: {solo_texto['n_palabras'].mean():.1f} palabras "
      f"({len(solo_texto):,} mensajes)")

tabla_p2 = pd.DataFrame({
    "mensajes con texto": con_palabras.groupby("grupo").size(),
    "promedio": con_palabras.groupby("grupo")["n_palabras"].mean(),
    "mediana": con_palabras.groupby("grupo")["n_palabras"].median(),
    "promedio sin repetidos": no_repetidos.groupby("grupo")["n_palabras"].mean(),
}).loc[orden]
tabla_p2.round(1)

En el boxplot de la derecha entran todos los usuarios con al menos 5 mensajes de texto;
con menos, el promedio depende de uno o dos mensajes y no dice mucho de cómo escribe esa
persona. Uso otra vez escala logarítmica porque unos pocos usuarios escriben mensajes muy
largos y aplastarían la caja.

In [ ]:
promedio_usuario = con_palabras.groupby("usuario")["n_palabras"].mean()
mensajes_texto = con_palabras.groupby("usuario").size()
promedio_5 = promedio_usuario[mensajes_texto >= 5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [3, 2]})
barras(ax1, tabla_p2["promedio"], "Palabras promedio por mensaje de texto",
       "Palabras por mensaje", formato="{:.1f}")
ax1.axvline(con_palabras["n_palabras"].mean(), color=GRIS, linestyle="--", zorder=0,
            label=f"Promedio del grupo ({con_palabras['n_palabras'].mean():.1f})")
ax1.legend(loc="lower right")

sns.boxplot(x=promedio_5.values, ax=ax2, color=AZUL, log_scale=True, width=0.4)
ax2.set_title(f"Promedio por usuario ({len(promedio_5)} usuarios con 5 o más mensajes)")
ax2.set_xlabel("Palabras por mensaje (escala logarítmica)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p2_palabras_por_mensaje.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Mediana del promedio por usuario: {promedio_5.median():.1f} palabras")

### Interpretación

Un mensaje con texto tiene en promedio 11.6 palabras, y el usuario típico (la mediana del
boxplot) escribe unas 10 por mensaje: son mensajes cortos, como descripciones de cuentas o
precios. Entre los 15 más activos, Robin (22.9) y Hulk (21.0) escriben los mensajes más
largos, y Flash (2.9) y Wolverine (3.2) los más cortos. Los repetidos cambian algunos
casos: Hulk solo publica anuncios repetidos y Batman sube de 11.4 a 22.3 palabras al
quitarlos. Si solo se cuentan los mensajes únicamente de texto, sin las descripciones de
fotos, el promedio del grupo sube de 11.6 a 13.0 palabras, así que las descripciones son un
poco más cortas que los mensajes normales.

## 5. ¿Quién envía más palabras, emojis y stickers?

Para cada usuario sumo todas sus palabras, emojis y stickers. Los rankings incluyen a todos
los usuarios (menos "Cuenta eliminada"), porque quien más escribe no siempre es quien más
mensajes manda. Como los anuncios repetidos inflan mucho el conteo de palabras, muestro ese
ranking con y sin ellos. Un sticker es un mensaje cuyo `tipo_contenido` es "sticker".

In [ ]:
usuarios["palabras_sin_repetir"] = usuarios["n_palabras"].where(~usuarios["es_repetido"], 0)
usuarios["es_sticker"] = usuarios["tipo_contenido"] == "sticker"

columnas = {
    "n_palabras": "palabras",
    "palabras_sin_repetir": "palabras sin repetidos",
    "n_emojis": "emojis",
    "es_sticker": "stickers",
}
totales = usuarios.groupby("usuario")[list(columnas)].sum().rename(columns=columnas)

# Los 15 más activos y "Otros"
tabla_p3 = usuarios.groupby("grupo")[list(columnas)].sum().rename(columns=columnas)
tabla_p3.loc[orden]

Las gráficas muestran los 10 primeros de todo el grupo en palabras y emojis. Como los
stickers son muy pocos, en ese panel solo aparecen los usuarios con al menos 3 stickers.

In [ ]:
stickers_3 = totales["stickers"][totales["stickers"] >= 3].sort_values(ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
barras(axes[0, 0], totales["palabras"].nlargest(10), "Palabras (con repetidos)", "Palabras")
barras(axes[0, 1], totales["palabras sin repetidos"].nlargest(10), "Palabras sin repetidos", "Palabras")
barras(axes[1, 0], totales["emojis"].nlargest(10), "Emojis", "Emojis")
barras(axes[1, 1], stickers_3, "Stickers (usuarios con 3 o más)", "Stickers")

fig.suptitle("Usuarios que más palabras, emojis y stickers envían", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "p3_palabras_emojis_stickers.png", dpi=120, bbox_inches="tight")
plt.show()

Cada sticker de Telegram lleva un emoji asociado, que el HTML exportado sí conserva. La
tabla muestra los más comunes.

In [ ]:
# Emojis asociados a los stickers (todo el grupo)
print(f"Stickers en total: {(df['tipo_contenido'] == 'sticker').sum()} "
      f"({100 * (df['tipo_contenido'] == 'sticker').mean():.1f} % de los mensajes)")
print(f"Usuarios que enviaron al menos un sticker: {(totales['stickers'] > 0).sum()}")
df["sticker_emoji"].value_counts().head(5).rename_axis("emoji").to_frame("stickers")



### Interpretación

Con los anuncios incluidos, Hulk es quien más palabras envía (15,435), pero todas vienen de
un mismo anuncio repetido; sin repetidos, el primero es Hawkeye (4,221), seguido de Thor
(3,436). En emojis gana Stargirl (923), que ni siquiera está entre los 15 más activos, y
Hulk vuelve a quedar segundo porque su anuncio lleva un emoji. Los stickers son solo el
0.3 % de los mensajes (114) y los envían 49 usuarios; Cable_2 manda 32, casi un tercio del
total, y el emoji más asociado a los stickers es 🌟. En resumen, los rankings de palabras y
emojis dependen mucho de los anuncios, y los stickers casi no se usan para conversar.

## 6. ¿Qué días y a qué horas se envían más mensajes?

Telegram guardó las horas en UTC-07:00, la zona horaria de la computadora con la que
exporté el chat y
no la de los miembros. Por eso en la sección 2 las pasé a hora de Moscú (UTC+3, sin horario
de verano), y con esa hora saco tanto el día de la semana como la hora del día. El cambio
también mueve el día: un mensaje enviado a las 20:00 en UTC-07:00 cae a las 6:00 del día
siguiente en Moscú.

Esto es un supuesto. El grupo está en ruso, pero puede tener miembros en otras zonas
horarias, como Siberia, Kazajistán o Ucrania, así que los resultados describen el ritmo del
grupo visto desde Moscú y no la hora local de cada persona. Aquí cuento todos los mensajes,
incluidos los de "Cuenta eliminada" y los anuncios repetidos, porque la pregunta es sobre la
actividad del grupo. Como el periodo no tiene exactamente el mismo número de lunes, martes,
etc., comparo el promedio de mensajes por día y no el total.

In [ ]:
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
dia = df["timestamp"].dt.dayofweek  # 0 = lunes
hora = df["timestamp"].dt.hour

# Todos los días del periodo, tengan o no mensajes
calendario = pd.date_range(df["timestamp"].min().normalize(),
                           df["timestamp"].max().normalize(), freq="D")
dias_en_periodo = pd.Series(calendario.dayofweek).value_counts()

tabla_p4 = pd.DataFrame({
    "mensajes": dia.value_counts(),
    "días en el periodo": dias_en_periodo,
    "promedio por día": dia.value_counts() / dias_en_periodo,
    "promedio sin repetidos": dia[~df["es_repetido"]].value_counts() / dias_en_periodo,
}).sort_index().set_axis(DIAS)

print(f"Periodo: del {calendario[0]:%d/%m/%Y} al {calendario[-1]:%d/%m/%Y} ({len(calendario)} días)")
tabla_p4.round(1)

In [ ]:
promedio_dia = len(df) / len(calendario)
promedio_hora = hora.value_counts().sort_index() / len(calendario)

print(f"Hora con más mensajes: {promedio_hora.idxmax()}:00 "
      f"({promedio_hora.max():.1f} mensajes en promedio)")
print(f"Hora con menos mensajes: {promedio_hora.idxmin()}:00 "
      f"({promedio_hora.min():.1f} mensajes en promedio)")
print(f"Mensajes enviados entre las 10:00 y las 23:59: {100 * hora.between(10, 23).mean():.1f} %")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 3]})

ax1.bar(DIAS, tabla_p4["promedio por día"], color=AZUL)
ax1.bar_label(ax1.containers[0], fmt="{:.0f}", padding=3, fontsize=9,
              bbox=dict(facecolor="white", edgecolor="none", pad=1))
ax1.axhline(promedio_dia, color=GRIS, linestyle="--", zorder=0,
            label=f"Promedio general ({promedio_dia:.0f})")
ax1.set_ylim(0, tabla_p4["promedio por día"].max() * 1.2)
ax1.set_title("Mensajes promedio por día de la semana")
ax1.set_ylabel("Mensajes por día")
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)
plt.setp(ax1.get_xticklabels(), rotation=30, ha="right")

ax2.bar(promedio_hora.index, promedio_hora.values, color=AZUL)
ax2.set_title("Mensajes promedio por hora del día")
ax2.set_xlabel("Hora (Moscú)")
ax2.set_ylabel("Mensajes por hora")
ax2.set_xticks(range(0, 24, 2))
ax2.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p4_dias_y_horas.png", dpi=120, bbox_inches="tight")
plt.show()

El heatmap cruza el día de la semana con la hora del día. Cada celda es el total de
mensajes que se enviaron en ese día y esa hora durante todo el periodo.

In [ ]:
conteo = pd.crosstab(dia, hora).set_axis(DIAS, axis=0)

fig, ax = plt.subplots(figsize=(13, 4.5))
sns.heatmap(conteo, cmap="Blues", linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "Mensajes"})
ax.set_title("Mensajes por día de la semana y hora del día")
ax.set_xlabel("Hora (Moscú)")
ax.set_ylabel("")
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "p4_heatmap_dia_hora.png", dpi=120, bbox_inches="tight")
plt.show()

### Interpretación

Por día de la semana no hay un patrón claro: el jueves es el día con más mensajes (92 en
promedio) y el miércoles el de menos (85), una diferencia de apenas 8 %, y el fin de semana
tiene casi el mismo promedio que los días entre semana, con o sin anuncios repetidos. Por
hora, en cambio, el ritmo es muy marcado: la actividad casi desaparece entre las 2:00 y las
6:00 de Moscú, sube desde las 9:00 y se mantiene alta de las 10:00 a las 23:00, con un pico a
las 15:00; en ese horario se envía el 80 % de los mensajes. El heatmap muestra que todos los
días siguen ese mismo ritmo, y las celdas más oscuras son el domingo y el jueves a las 15:00.
La actividad que sigue a medianoche puede venir de jugadores que se desvelan o de miembros
al este de Moscú, lo que recuerda que usar esa zona horaria es un supuesto.